# Manipulating EDAM with owlready2

## Loading RDF data

### Install and load python libraries

In [59]:
%pip install rdflib
%pip install pyyaml
%pip install pandas
%pip install owlready2


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [77]:
from rdflib import Graph, Literal, ConjunctiveGraph
#import rdflib
import os
import pandas as pd
from owlready2 import *



### Dumps knowledge graph

In [79]:
edam = 'https://github.com/edamontology/edamontology/raw/main/EDAM_dev.owl'

bioconda_rdf = 'https://raw.githubusercontent.com/rioualen/RSEc_SPARQL_endpoint/refs/heads/main/data/bioconda-dump.ttl'
biocontainers_rdf = 'https://raw.githubusercontent.com/rioualen/RSEc_SPARQL_endpoint/refs/heads/main/data/biocontainers-dump.ttl'
bioschemas_rdf = 'https://raw.githubusercontent.com/rioualen/RSEc_SPARQL_endpoint/refs/heads/main/data/bioschemas-dump.ttl'
debian_rdf = 'https://raw.githubusercontent.com/rioualen/RSEc_SPARQL_endpoint/refs/heads/main/data/debian-dump.ttl'
galaxytools_rdf = 'https://raw.githubusercontent.com/rioualen/RSEc_SPARQL_endpoint/refs/heads/main/data/galaxy-dump.ttl'

kg = ConjunctiveGraph()
kg.parse(bioconda_rdf)
kg.parse(biocontainers_rdf)
kg.parse(bioschemas_rdf)
kg.parse(debian_rdf)
kg.parse(galaxytools_rdf)
kg.parse(edam, format='xml')
print(str(len(kg)) + ' triples in the kg after parsing all datasets')


/var/folders/hc/5k1hmm_j4bx5_h201tn5krs80000gq/T/ipykernel_57268/3515556108.py:9: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  kg = ConjunctiveGraph()


827930 triples in the kg after parsing all datasets


### EDAM knowledge graph

In [61]:
#edam_version = 'https://edamontology.org/EDAM_1.25.owl'
#edam_version = 'https://raw.githubusercontent.com/edamontology/edamontology/master/EDAM_dev.owl'
edam = 'https://github.com/edamontology/edamontology/raw/main/EDAM_dev.owl'

#onto = get_ontology(edam).load()


edam_kg = ConjunctiveGraph()
edam_kg.parse(edam, format='xml')
print(str(len(edam_kg)) + ' triples in the edam_kg after parsing all datasets')


/var/folders/hc/5k1hmm_j4bx5_h201tn5krs80000gq/T/ipykernel_57268/2465588208.py:8: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  edam_kg = ConjunctiveGraph()


39057 triples in the edam_kg after parsing all datasets


## Owlready

### Load EDAM with owlready2

In [28]:

# Define path to local ontology file in case IRI fails 
onto_path.append("/Users/rioualen/Desktop/Work/_Git/EDAM/edamontology")

# Load the ontology file from IRI - filename inferred from the IRI provided
onto = get_ontology(edam).load()
#onto_local = get_ontology("EDAM_dev.owl").load()

### Function to calculate class depth

In [26]:
def get_class_depth(cls):
    """Return the depth of a class in the ontology hierarchy."""
    if cls is None:
        return 0
    # Get the parent classes (superclasses)
    parents = cls.is_a
    if not parents:
        return 0  # Root class (e.g., owl:Thing)
    # Recursively calculate the maximum depth of all parent classes
    return 1 + max(get_class_depth(parent) for parent in parents)

### Example class depth

In [29]:
# Get the class from the ontology
topic_0003 = onto.topic_0003
topic_3391 = onto.topic_3391

print(f"Class: {topic_3391.name}")

depth = get_class_depth(topic_3391)
print(f"The depth of {topic_3391.name} is: {depth}")

Class: topic_3391
The depth of topic_3391 is: 3


## Rdflib

In [ ]:

#edam_version = 'https://github.com/edamontology/edamontology/raw/main/EDAM_dev.owl'

#kg = ConjunctiveGraph()
#kg.parse(edam_version, format='xml')

def getEdamLabelsFromUris(edam_uris) -> list :
  """
  Get EDAM labels from EDAM URIs.
  """

  res = []

  for uri in edam_uris:
    query="""
    PREFIX edam: <http://edamontology.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?label ?entity WHERE {
        edam:%s rdfs:label ?label .
    }
    """%(uri)

    q = edam_kg.query(query)
    #print(q)
    for r in q:
        lab = r['label']
        #print(lab)
        #url.rsplit('/', 1)
        res.append(f'{lab}')

  return res

In [57]:
## Example usage
#edam_uris = ['http://edamontology.org/topic_3391', 'http://edamontology.org/topic_0003']
edam_uris = ['topic_3391', 'topic_0003']

edam_labels = getEdamLabelsFromUris(edam_uris)

print(edam_labels)

['Omics', 'Topic']


## Pandas

In [80]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?application ?biotools_ID 
WHERE {
    ?application schema:identifier ?biotools_ID .
    FILTER STRSTARTS(STR( ?biotools_ID ), "https://bio.tools")

}
ORDER BY ?biotools_ID
"""

results = kg.query(query)

for r in results :
    print(f"{r['application']} - {r['biotools_ID']}")



https://github.com/bioconda/bioconda-recipes/tree/master/recipes/abricate - https://bio.tools/ABRicate
https://github.com/galaxyproject/tools-iuc/tree/master/tools/abricate - https://bio.tools/ABRicate
https://github.com/bioconda/bioconda-recipes/tree/master/recipes/admixtools - https://bio.tools/AdmixTools
https://biocontainers.pro/tools/admixtools - https://bio.tools/AdmixTools
https://github.com/bioconda/bioconda-recipes/tree/master/recipes/arriba - https://bio.tools/Arriba
https://github.com/bioconda/bioconda-recipes/tree/master/recipes/augur - https://bio.tools/Augur
https://github.com/bioconda/bioconda-recipes/tree/master/recipes/beagle - https://bio.tools/BEAGLE
https://biocontainers.pro/tools/beagle - https://bio.tools/BEAGLE
https://github.com/bioconda/bioconda-recipes/tree/master/recipes/bufet - https://bio.tools/BUFET
https://biocontainers.pro/tools/bufet - https://bio.tools/BUFET
https://github.com/bioconda/bioconda-recipes/tree/master/recipes/bamutil - https://bio.tools/Ba

In [114]:

results = kg.query(query)

d = []
for r in results :
    #print(f"{r['application']} - {r['biotools_ID']}")
    d.append((r['application'], r['biotools_ID']))

df = pd.DataFrame(d, columns=('application', 'biotools_ID'))
df.head()

df2 = df.groupby(["biotools_ID"]).count()
df2.head()



,application
biotools_ID,
https://bio.tools/ABRicate,2
https://bio.tools/AdmixTools,2
https://bio.tools/Arriba,1
https://bio.tools/Augur,1
https://bio.tools/BEAGLE,2


In [ ]:
print(os.getcwd())
filename = "results_queries/applications_biotoolsIDs.tsv"
df = pd.read_csv(filename, sep='\t')

print('Table has ' + str(df.shape[1]) + ' columns and ' + str(df.shape[0]) + ' rows.')
#print(df.head())


df2 = pd.DataFrame(df, columns=('application', 'biotools_ID'))
#df2.head()

df3 = df2.groupby(["biotools_ID"]).count()
#df3.head()





/Users/rioualen/Desktop/Document/_Git/research-software-ecosystem/RSEc_SPARQL_endpoint/notebook
Table has 2 columns and 4484 rows.


TypeError: 'DataFrame' object is not callable